# NLP 2026
# Lab 3: Classification and BERT

Have you ever read a movie review and wondered:

```“Is this review actually positive or negative?” 🤔```

In this lab, you will build your own sentiment analysis tool using Natural Language Processing (NLP)! Your goal is to automatically classify movie reviews into one of two categories:

✅ Positive

❌ Negative

We will approach this as a binary classification task and you will experiment with increasingly powerful methods — from classic machine learning to modern neural networks based on transformers 🚀

### 🎯 Learning Goals ###

By completing this lab, you should be able to:

- Formulate sentiment analysis as a binary classification problem

- Design and evaluate hand-crafted text features

- Implement a Bag-of-Words representation

- Apply and evaluate Logistic Regression and alternative classifiers

- Understand how BERT tokenization and embeddings work

- Extract sentence representations using: ```CLS``` token, mean token pooling

- Compare classical ML and transformer-based methods

- Critically analyze evaluation metrics beyond accuracy 📊



### Score breakdown

| Exercise            | Points |
|---------------------|--------|
| [Exercise 1](#e1)   | 1      |
| [Exercise 2](#e2)   | 5      |
| [Exercise 3](#e3)   | 5      |
| [Exercise 4](#e4)   | 5      |
| [Exercise 5](#e5)   | 5      |
| [Exercise 6](#e6)   | 5      |
| [Exercise 7](#e7)   | 3      |
| [Exercise 8](#e8)   | 6      |
| [Exercise 9](#e9)   | 5      |
| [Exercise 10](#e10) | 5      |
| [Exercise 12](#e12) | 5      |
| [Exercise 13](#e13) | 10     |
| Total               | 60     |

This score will be scaled down to 0.6 and that will be your final lab score.

### 📌 **Instructions for Delivery** (📅 **Deadline: 23/Feb 18:00**, 🎭 *wildcards possible*)

✅ **Submission Requirements**
+ 📄 You need to submit a **notebook** 📓 with the code, appropriate comments and figures in all questions. Make sure to have a mix of code (some explanations needed if not clear what you implement), figures to support the answers or your claims and proper amount of text to explain your reasoning, answer etc.
+ ⚡ Make sure that **all cells are executed properly** ⚙️ and that **all figures/results/plots** 📊 you include in the report are also visible in your **executed notebook**.
+ You can work on Google Collab (or other environments), but you need to make sure that your delivered notebook is executed properly.

✅ **Collaboration & Integrity**
+ 🗣️ While you may **discuss** the lab with others, you must **write your solutions with your group only**. If you **discuss specific tasks** with others, please **include their names** below.
+ 📜 **Honor Code applies** to this lab. For more details, check **Syllabus §7.2** ⚖️.
+ 📢 **Mandatory Disclosure**:
   - Any **websites** 🌐 (e.g., **Stack Overflow** 💡) or **other resources** used must be **listed and disclosed**.
   - Any **GenAI tools** 🤖 (e.g., **ChatGPT**) used must be **explicitly mentioned**.
   - 🚨 **Failure to disclose these resources is a violation of academic integrity**. See **Syllabus §7.3** for details.

## 0. Setup

We first install the scikit-learn library [Scikit-learn](https://scikit-learn.org/stable/). We will use its classification models.

In [44]:
pip install -U scikit-learn

Note: you may need to restart the kernel to use updated packages.


We will need [PyTorch](https://pytorch.org/) installed. It is a very popular deep learning library that offers modularized versions of many of the sequence models we discussed in class. It's an important tool that you may want to practice further if you want to dive deeper into NLP, since much of the current academic and industrial research uses it.

Some resources to look further are given below.

* [Documentation](https://pytorch.org/docs/stable/index.html) (We will need this soon)

* [Installation Instructions](https://pytorch.org/get-started/locally/)

* [Quickstart Tutorial](https://pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html)

The cell below should install the library:

In [45]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


The last bit we need is the huggingface transformers library (here is the documentation [https://huggingface.co/docs/transformers/en/index](https://huggingface.co/docs/transformers/en/index)). Transformers are one of the most influential architectures in handling sequences (not only in language). As we discussed in lectures, they excel at taking into account context (which is the salt-and-pepper of NLP) with mechansisms such as self-attetion, which allows them to weigh the importance of different words in a sentence. If you want to know more, revisit the course material (slides and textbook).

We already used huggingface datasets in previous labs and huggingface transformers integrates nicely with that. Apart from the ease of use, huggingface is also providing pre-trained models of different kinds. The list can be found [here](https://huggingface.co/models) ([https://huggingface.co/models](https://huggingface.co/models)). The following line should be enough to install huggingface transformers library:

In [46]:
pip install transformers

Note: you may need to restart the kernel to use updated packages.


Here, we import the libraries:

In [47]:
import re
from collections import Counter

import datasets
import numpy as np
import torch
import tqdm
import transformers
import requests
import string
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from torch.utils.data.dataloader import DataLoader

## 1. Loading the Dataset

We will work with the IMDB dataset [https://huggingface.co/datasets/stanfordnlp/imdb](https://huggingface.co/datasets/stanfordnlp/imdb). It contains the reviews and a label that indicates whether the review is positive or not (the neutral reviews have been filtered out). You can read the paper [here](https://aclanthology.org/P11-1015/).

In [48]:
dataset = datasets.load_dataset('stanfordnlp/imdb', split=['train', 'test'])
print(dataset)

[Dataset({
    features: ['text', 'label'],
    num_rows: 25000
}), Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})]


Notice that the dataset has been loaded as a list of two datasets. They are the `train` and `test` splits that we asked for.
We will use the validation subset to tune the parameters. So, let's split the `train` subset and create a `DatasetDict` object:

In [49]:
train_valid_split = dataset[0].train_test_split(5000)
dataset = datasets.DatasetDict({
    'train': train_valid_split['train'].shuffle(),
    'validation': train_valid_split['test'].shuffle(),
    'test': dataset[1].shuffle(),
})
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})


We can print several examples from the `train` dataset:

In [50]:
for i in range(20):
    print('i', i)
    print(dataset['train'][i]['text'])
    print(dataset['train'][i]['label'])
    print()

i 0
I'd give it a 2/10.<br /><br />I was really, really disappointed.<br /><br />The storyline was poorly developed, for instance, the incidents were too short and brief, hence the moral was not clearly brought out. I thought Brenda Song did a fine job, but Shin Koyamada seemed to have a difficult time handling his role. I could see the need to put in western elements in the show, however, there are certain parts, where Chinese elements were needed too! The villain for example. His physical appearance resembled a robot, instead of something out of the Chinese culture. The final and the worst flaw, were the incorrect, and distorted facts placed in the show.<br /><br />Others may point out that this is a 'Kid's show' and hence, there is no need for the such high standards. However, there are other Disney shows, such as Mu Lan, which have been much better in terms of story development and presentation.<br /><br />In conclusion, I feel that Disney movies should be better researched and bet

Let's extract the labels from the dataset. We will use them to train and evaluate our classifiers.

In [51]:
y_train = dataset['train']['label']
print(y_train)
y_valid = dataset['validation']['label']
print(len(y_valid))
print(y_valid)

Column([0, 1, 1, 1, 1, ...])
5000
Column([0, 0, 0, 1, 1, ...])


<a name='e1'></a>
#### Exercise 1: Cleaning the text

(1p) In this exercise you should clean the text in the dataset. This is the same step we saw in the previous labs.

If you think this step is not necessary in this use case, you can skip this step, but make sure to justify your decision.

In [52]:
import html
def clean(text):
    """
    Cleans the text
    Args:
        text: a string that will be cleaned

    Returns: the cleaned text

    """

    # Empty text
    if text == '':
        return text

    ### YOUR CODE HERE
    if text == '':
        return text

    text = text.replace("\xa0", " ")
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = text.replace("’", "'")
    punct_pattern = f"[{re.escape(string.punctuation)}]"
    text = re.sub(punct_pattern, r" \g<0> ", text)

    text = re.sub(r'\s+', ' ', text).strip()
    # no stopword removal since BERT works with natural language (we would remove syntactic meaning and function words that help attention patterns, same goes for contractions)
    # and no lowering ("BORING" is an emotion indicator) or punctuation removal (we need it for exercise 2)
    ### YOUR CODE ENDS HERE

    return text


def clean_example(example):
    """
    Applies the clean() function to the example from the Dataset
    Args:
        example: an example from the Dataset

    Returns: update example with cleaned 'text' column

    """
    example['text'] = clean(example['text'])
    return example


dataset = dataset.map(clean_example, desc="Cleaning")
print(dataset)

Cleaning:   0%|          | 0/20000 [00:00<?, ? examples/s]

Cleaning:   0%|          | 0/5000 [00:00<?, ? examples/s]

Cleaning:   0%|          | 0/25000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})


## 2. Hand-crafted Features

<a name='e2'></a>
#### Exercise 2: Hand-crafted features

(5p) Write your own hand-crafted feature extraction function. Include at least these types of features:
- length of the text,
- number of different punctuation characters,
- number of positive and negative words.

In [53]:

url_positive = 'https://drive.google.com/uc?export=download&id=1IinfohCnPfJhPoJw_HCPnscjvyNdzLKS'
url_negative = 'https://drive.google.com/uc?export=download&id=1t4Gh6ctrQgWTWJlba4pIFRD3_MBG8pyn'

# create lexicon set written by AI

def create_lexicon_set(url):
    # Fetch content from Google Drive
    r = requests.get(url)
    # Using latin-1 is safer for these specific sentiment files
    content = r.content.decode('latin-1')

    lexicon = set()
    for line in content.splitlines():
        # Clean the line and skip comments (;) or empty lines
        clean_line = line.strip()
        if clean_line and not clean_line.startswith(';'):
            lexicon.add(clean_line.lower())
    return lexicon


In [54]:
### YOUR CODE HERE
# you can define the positive and negative words here

# we defined positive and negative words using opinion lexicon from the kaggle files
# found here https://www.kaggle.com/datasets/nltkdata/opinion-lexicon

pos_lexicon = create_lexicon_set(url_positive)
neg_lexicon = create_lexicon_set(url_negative)


def calculate_features(text):
    features = []
    ### YOUR CODE HERE
    # num of words in a text
    features.append((text.count(' ')+1))
    # number of punctuation chars
    punctuation_chars = 0
    # num of positive words
    pos_words = 0
    neg_words = 0
    for token in text.split():
      if token in pos_lexicon:
        pos_words += 1
      elif token in neg_lexicon:
        neg_words += 1
      elif token in string.punctuation:
        punctuation_chars += 1
    features.append(punctuation_chars)
    features.append(pos_words)
    features.append(neg_words)
    ### YOUR CODE ENDS HERE
    return np.array(features, dtype=float)


### YOUR CODE ENDS HERE

The function below will apply your feature extraction implementation to a specified dataset.

In [55]:
def calculate_features_dataset(dataset, features_fn):
    all_features = []
    for e in tqdm.tqdm(dataset, desc='Extracting features'):
        text = e['text']
        features = features_fn(text)
        all_features.append(features)
    all_features = np.array(all_features, dtype=float)
    return all_features

And we can obtain the features for the `train` and `validation` splits. Later you will need to do the same for the `test` subset.

In [56]:
X_train = calculate_features_dataset(dataset['train'], calculate_features)
X_valid = calculate_features_dataset(dataset['validation'], calculate_features)

Extracting features: 100%|███████████████| 5000/5000 [00:00<00:00, 18166.71it/s]


In [57]:
print(X_train)

[[220.  33.   7.   7.]
 [881. 108.  40.  32.]
 [159.  22.   3.   5.]
 ...
 [101.  26.   3.   3.]
 [231.  43.   9.   9.]
 [ 69.   9.   6.   3.]]


### 2.1 Classification

In this section, we will create and train a logistic regression classifier. We will train it on the `train` subset and evaluate on the `validation` split. Later, you will do a final comparison between methods on the `test` subset, but it is important to avoid it when tuning the methods.

In [58]:
classifier = LogisticRegression(solver='lbfgs', max_iter=1000)
classifier.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

Let's check the performance on the training data:

In [59]:
print('Features results: train')
pred_train = classifier.predict(X_train)
print(accuracy_score(y_train, pred_train))

Features results: train
0.7276


... and the validation dataset:

In [60]:
print('Features results: validation')
pred_valid = classifier.predict(X_valid)
print(accuracy_score(y_valid, pred_valid))

Features results: validation
0.7266


<a name='e3'></a>
#### Exercise 3: Improving the features

(5p) Iteratively improve your hand-crafted features. Think about what information from the review might be useful for to predict the rating a person gave to the particular movie. You can also look into the expected format (or range) of features for the classifier.

Document the steps you tried (even if unsuccessful) and how they influenced the metrics. Try at least 3 modifications from your original implementation.

In [61]:
### YOUR CODE HERE
def calculate_features_enhanced(text):
  # [num_tokens, num_puncutation, num_pos, num_neg]
  features = calculate_features(text).tolist()
  polarity = (features[2] - features[3]) / (features[2] + features[3] + 1)
  features.append(polarity)
  return np.array(features, dtype=float)

### YOUR CODE ENDS HERE

In [62]:
def calculate_features_enhanced_v2(text):
  base = calculate_features_enhanced(text).tolist()
  tokens = text.split()
  word_tokens = [t for t in tokens if t not in string.punctuation]
  num_words = len(word_tokens)
  exclam = tokens.count("!")
  quest = tokens.count("?")
  repeat_punct = len(re.findall(r"[!?]{2,}", text))
  denom = num_words + 1e-6
  base += [exclam, quest, exclam/denom, quest/denom, repeat_punct]
  return np.array(base, dtype=float)

In [63]:
def calculate_features_enhanced_v3(text):
  base = calculate_features_enhanced_v2(text).tolist()
  tokens = text.split()
  uppercase_words = 0
  for token in tokens:
    if token.isupper() and len(token) >=3:
      uppercase_words += 1
  base.append(uppercase_words)
  return np.array(base, dtype = float)


In [64]:
def calculate_features_enhanced_v4(text):
    # AI proposed version to check quality of ours
    # base = [num_tokens, num_punctuation, num_pos, num_neg]
    base = calculate_features(text).tolist()
    tokens = text.split()
    total_tokens = base[0]

    # 1. EMOTION: Shouting and Exclamations (Modification #1)
    # We check len > 2 so we don't count "I" or "A" as shouting
    shouting = sum(1 for w in tokens if w.isupper() and len(w) > 2)
    exclamations = text.count('!')

    # 2. DENSITY: Ratios (Modification #2)
    # Using 1e-6 to avoid division by zero
    denom = total_tokens + 1e-6
    ratio_pos = base[2] / denom
    ratio_neg = base[3] / denom
    # Subjectivity: How much of the review is actually sentiment-heavy?
    subjectivity = (base[2] + base[3]) / denom

    # 3. DISTRIBUTION: Log-scaling the raw counts (Modification #3)
    # This prevents long reviews from having 100x the influence of short ones
    log_counts = [np.log1p(x) for x in base]

    # 4. Final Feature Vector
    # [log_len, log_punc, log_pos, log_neg, r_pos, r_neg, shout, bang, sub, net_score]
    enhanced = log_counts + [
        ratio_pos,
        ratio_neg,
        shouting,
        exclamations,
        subjectivity,
        (ratio_pos - ratio_neg)
    ]

    return np.array(enhanced, dtype=float)


In [65]:
classifier = LogisticRegression(max_iter=2000)
X_train = calculate_features_dataset(dataset['train'], calculate_features_enhanced)
X_valid = calculate_features_dataset(dataset['validation'], calculate_features_enhanced)
classifier.fit(X_train, y_train)
print('Features results v1: train')
pred_train = classifier.predict(X_train)
print(accuracy_score(y_train, pred_train))
print('Features results v1: validation')
pred_valid = classifier.predict(X_valid)
print(accuracy_score(y_valid, pred_valid))

Extracting features: 100%|███████████████| 5000/5000 [00:00<00:00, 18719.02it/s]


Features results v1: train
0.7316
Features results v1: validation
0.7282


In [66]:
classifier_handcrafted_final = LogisticRegression(max_iter=2000)
X_train = calculate_features_dataset(dataset['train'], calculate_features_enhanced_v2)
X_valid = calculate_features_dataset(dataset['validation'], calculate_features_enhanced_v2)
classifier.fit(X_train, y_train)
print('Features results v2: train')
pred_train = classifier.predict(X_train)
print(accuracy_score(y_train, pred_train))
print('Features results v2: validation')
pred_valid = classifier.predict(X_valid)
print(accuracy_score(y_valid, pred_valid))

Extracting features: 100%|███████████████| 5000/5000 [00:00<00:00, 11066.04it/s]


Features results v2: train
0.7379
Features results v2: validation
0.7388


In [67]:
classifier = LogisticRegression(max_iter=2000)
X_train = calculate_features_dataset(dataset['train'], calculate_features_enhanced_v3)
X_valid = calculate_features_dataset(dataset['validation'], calculate_features_enhanced_v3)
classifier.fit(X_train, y_train)
print('Features results v3: train')
pred_train = classifier.predict(X_train)
print(accuracy_score(y_train, pred_train))
print('Features results v3: validation')
pred_valid = classifier.predict(X_valid)
print(accuracy_score(y_valid, pred_valid))

Extracting features: 100%|████████████████| 5000/5000 [00:00<00:00, 9852.04it/s]


Features results v3: train
0.7381
Features results v3: validation
0.7384


In [68]:
from sklearn.preprocessing import StandardScaler

X_train_v4 = calculate_features_dataset(dataset['train'], calculate_features_enhanced_v4)
X_valid_v4 = calculate_features_dataset(dataset['validation'], calculate_features_enhanced_v4)

scaler = StandardScaler()
classifier = LogisticRegression(max_iter=2000)
X_train_scaled = scaler.fit_transform(X_train_v4)
X_valid_scaled = scaler.transform(X_valid_v4)

classifier.fit(X_train_scaled, y_train)
pred_v4_train = classifier.predict(X_train_scaled)
pred_v4_valid = classifier.predict(X_valid_scaled)

print(f"Features results v4: train {accuracy_score(y_train, pred_v4_train)}")
print(f"Features results v4: validation {accuracy_score(y_valid, pred_v4_valid)}")


Extracting features: 100%|███████████████| 5000/5000 [00:00<00:00, 16029.80it/s]


Features results v4: train 0.7333
Features results v4: validation 0.7328


--- YOUR ANSWERS HERE

1. Adding Polarization improved the validation set accuracy by around 1.5-2 percentage points.

2. Adding the count of exclamation and question marks improved the accuracy by around 0.5 percentage point on validation set.

3. Adding the count of capitalized words did not improve the validation set accuracy and the accuract stayed the same.

4. Applying Z-score normalization and log-scaling the raw counts did not significantly improve the validation accuracy.

The best accuracy on the validation set out of all our trials was 0,7494.

<a name='e4'></a>
#### Exercise 4: Improving the evaluation

(5p) In the previous cells, we only looked at the accuracy of predictions. Investigate which other metrics might be better for our case. You can check the documentation of scikit-learn for evaluation metrics ([https://scikit-learn.org/stable/api/sklearn.metrics.html#classification-metrics](https://scikit-learn.org/stable/api/sklearn.metrics.html#classification-metrics)). Give reasons why the metrics you try can be more informative than raw accuracy score.

Decide which evaluation metric(s) is most suitable for our use-case and give reasons why. Test your features-based classifier and all further classifiers on that metric (apart from the accuracy score).

In [69]:
### YOUR CODE HERE
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    confusion_matrix, matthews_corrcoef, classification_report
)

def evaluate(y_true, y_pred, name=""):
    print(name)
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("F1:", f1_score(y_true, y_pred, average="macro"))
    print("Balanced Accuracy:", balanced_accuracy_score(y_true, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print("\nPer-class report:\n", classification_report(y_true, y_pred, digits=4))
    print()
X_train = calculate_features_dataset(dataset['train'], calculate_features_enhanced_v3)
X_valid = calculate_features_dataset(dataset['validation'], calculate_features_enhanced_v3)

classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train, y_train)

pred_training = classifier.predict(X_train)
evaluate(y_train, pred_training, "Training")

pred_valid = classifier.predict(X_valid)
evaluate(y_valid, pred_valid, "Validation")
### YOUR CODE ENDS HERE

Extracting features: 100%|████████████████| 5000/5000 [00:00<00:00, 9927.77it/s]


Training
Accuracy: 0.7381
F1: 0.7380994866749939
Balanced Accuracy: 0.7381057524230097
Confusion Matrix:
 [[7367 2653]
 [2585 7395]]

Per-class report:
               precision    recall  f1-score   support

           0     0.7403    0.7352    0.7377     10020
           1     0.7360    0.7410    0.7385      9980

    accuracy                         0.7381     20000
   macro avg     0.7381    0.7381    0.7381     20000
weighted avg     0.7381    0.7381    0.7381     20000


Validation
Accuracy: 0.7384
F1: 0.7383979490399204
Balanced Accuracy: 0.7384248591909883
Confusion Matrix:
 [[1839  641]
 [ 667 1853]]

Per-class report:
               precision    recall  f1-score   support

           0     0.7338    0.7415    0.7377      2480
           1     0.7430    0.7353    0.7391      2520

    accuracy                         0.7384      5000
   macro avg     0.7384    0.7384    0.7384      5000
weighted avg     0.7384    0.7384    0.7384      5000




--- YOUR ANSWERS HERE

Because the classes are pretty much balanced and the errors are symmetrical (looking at the confusion matrix) there's no metric that's significantly better than simple accuracy by a large margin if we take into account the numeric values. However, in theory F1 is preferable as it evaluates precision and recall per class. The same can be said for balanced accuracy, it accounts for any class imbalance or majority class dominance. So, we'll work with balanced accuracy.

In [70]:
final_classifier_handcrafted = LogisticRegression(max_iter=1000)
X_train_final = calculate_features_dataset(dataset['train'], calculate_features_enhanced_v2)
X_valid_final = calculate_features_dataset(dataset['validation'], calculate_features_enhanced_v2)
final_classifier_handcrafted.fit(X_train_final, y_train)



Extracting features: 100%|███████████████| 5000/5000 [00:00<00:00, 10946.54it/s]


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

## 3. Bag-of-Words Classifier

Similar to the previous lab, we will use the classic bag-of-words representation as one of our embeddings. While it is simple and does not preserve the positions of words, it gives our classifier a lot of useful information.

<a name='e5'></a>
#### Exercise 5: Implementing BOW

(5p) Implement the BOW. In this exercise, we do not give you a rigid structure, so you can conjure your own. The two things your code should produce is the `token_to_id` dictionary, and `bag_of_words()` function that accepts a list of tokens, and the `token_to_id` dictionary while generating the BOW representation as a numpy array.

In [71]:
MAX_VOCAB_SIZE = 1000

In [72]:
def create_vocab(dataset, max_size: int = MAX_VOCAB_SIZE):
  vocab = Counter()
  for token in dataset['tokens']:
    vocab.update(token)
  vocab = vocab.most_common(max_size)
  vocab = [word for word, _ in vocab]
  return vocab

In [73]:
def create_token_to_id(vocab):
  return {word: i for i, word in enumerate(vocab)}

In [74]:
def tokenize(dataset):
  dataset['tokens'] = dataset['text'].split()
  return dataset
dataset = dataset.map(tokenize)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [75]:
#### YOUR CODE HERE



# The goal is to implement the `bag_of_words(tokens, token_to_id)` function similar to the previous lab.
# You might want to follow the steps:
# - tokenize the `text` column in the dataset,
# - extract the vocabulary from the tokens,
# - limit the vocabulary to `MAX_VOCAB_SIZE`,
# - calculate the `token_to_id` dictionary
# - implement the `bag_of_words(tokens, token_to_id)` function.

def bag_of_words(tokens, token_to_id):
    """
    Creates a bag-of-words representation of the sentence
    Args:
        tokens: a list of tokens
        token_to_id: a dictionary mapping each word to an index in the vocabulary

    Returns:: a numpy array of size vocab_size with the counts of each word in the vocabulary
    """
    bow = np.zeros(len(token_to_id))
    for token in tokens:
      if token in token_to_id:
        idx = token_to_id[token]
        bow[idx] += 1

    return bow

#### YOUR CODE ENDS HERE

In [76]:
vocab = create_vocab(dataset['train'])
token_to_id = create_token_to_id(vocab)

Here, we will use your implemented function to calculate the bag-of-words for each example in the `train` and `validation` subsets.

In [77]:
train_bows = []
for example in tqdm.tqdm(dataset['train'], desc='Calculating train BOWs'):
    train_bows.append(bag_of_words(example['tokens'], token_to_id))
train_bows = np.array(train_bows, dtype=float)

valid_bows = []
for example in tqdm.tqdm(dataset['validation'], desc='Calculating validation BOWs'):
    valid_bows.append(bag_of_words(example['tokens'], token_to_id))
valid_bows = np.array(valid_bows, dtype=float)


Calculating validation BOWs: 100%|████████| 5000/5000 [00:00<00:00, 6423.69it/s]


In [78]:
def get_bows(dataset, token_to_id_tmp):
  bows = []
  for example in tqdm.tqdm(dataset, desc='Calculating BOWs'):
    bows.append(bag_of_words(example['tokens'], token_to_id_tmp))
  bows = np.array(bows, dtype=float)
  return bows

Finally, we can train the classifier on the BOW representations and the labels in the `train` split.

In [79]:
classifier = LogisticRegression(solver='lbfgs', max_iter=1000)
print('Training classifier...')
classifier.fit(train_bows, y_train)

Training classifier...


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

Let's evaluate the classifier:

In [80]:
print('BOW results: train')
pred_train = classifier.predict(train_bows)
print(accuracy_score(y_train, pred_train))

print('BOW results: validation')
pred_valid = classifier.predict(valid_bows)
print(accuracy_score(y_valid, pred_valid))

BOW results: train
0.88185
BOW results: validation
0.848


<a name='e6'></a>
#### Exercise 6: Tuning the model

(5p) Try different values for the vocab size. Experiment with adding the hand-crafted features. Test the model on the evaluation metric of your choice (remember to use the validation split).

In [81]:
#### YOUR CODE HERE
from sklearn.feature_extraction.text import TfidfTransformer
from scipy.sparse import hstack, csr_matrix

candidates = [1000, 1500, 2000]
best_acc = 0
best = candidates[0]
best_vectors = None
for candidate in candidates:
  vocab = create_vocab(dataset['train'], candidate)
  token_to_id = create_token_to_id(vocab)
  tfidf = TfidfTransformer()
  train_bows = get_bows(dataset['train'], token_to_id)
  train_tfidf = tfidf.fit_transform(train_bows)
  X_train_combo = hstack([train_tfidf, csr_matrix(X_train_final)], format="csr")
  val_bows = get_bows(dataset['validation'], token_to_id)
  val_tfidf = tfidf.transform(val_bows)
  X_valid_combo = hstack([val_tfidf, csr_matrix(X_valid_final)], format="csr")
  classifier = LogisticRegression(solver='lbfgs', max_iter=8000)
  classifier_enhanced_features = LogisticRegression(solver='lbfgs', max_iter=8000)
  classifier_notfidf = LogisticRegression(solver='lbfgs', max_iter=8000)
  print(f"Training classifier with vocab size {candidate} ...")
  classifier.fit(train_tfidf, y_train)
  classifier_notfidf.fit(train_bows, y_train)
  classifier_enhanced_features.fit(X_train_combo, y_train)
  print('BOW results: train')
  pred_train = classifier.predict(train_tfidf)
  print(balanced_accuracy_score(y_train, pred_train))
  print('BOW results: valid')
  pred_val = classifier.predict(val_tfidf)
  pred_val_notfidf = classifier_notfidf.predict(val_bows)
  pred_val_enhanced = classifier_enhanced_features.predict(X_valid_combo)
  print("TF-IDF Validation")
  print(balanced_accuracy_score(y_valid, pred_val))
  print("No TF-IDF Validation")
  print(balanced_accuracy_score(y_valid, pred_val_notfidf))
  print("TF-IDF + handcrafted features - Validation")
  print(balanced_accuracy_score(y_valid, pred_val_enhanced))
  if balanced_accuracy_score(y_valid, pred_val) > best_acc:
    best_acc = balanced_accuracy_score(y_valid, pred_val)
    best = candidate
    best_vectors = train_tfidf



  # Results from some run. The validation set accuracy settled at vocab size 1500.
  # Vocab 500
  # BOW results: train
  # 0.84525
  # BOW results: valid
  # 0.832
  # Vocab 1000
  # BOW results: train
  # 0.879
  # BOW results: valid
  # 0.8594
  # Vocab 1500
  # BOW results: train
  # 0.89905
  # BOW results: valid
  # 0.8674
  # Vocab 2000
  # BOW results: train
  # 0.91315
  # BOW results: valid
  # 0.8668
  # Vocab 2500
  # BOW results: train
  # 0.9254
  # BOW results: valid
  # 0.8642

#### YOUR CODE ENDS HERE

Calculating BOWs: 100%|███████████████████| 5000/5000 [00:00<00:00, 6453.46it/s]


Training classifier with vocab size 1000 ...
BOW results: train
0.8733110932443731
BOW results: valid
TF-IDF Validation
0.855453149001536
No TF-IDF Validation
0.8479102662570404
TF-IDF + handcrafted features - Validation
0.8599174347158218


Calculating BOWs: 100%|███████████████████| 5000/5000 [00:00<00:00, 6274.27it/s]


Training classifier with vocab size 1500 ...
BOW results: train
0.8867123468493874
BOW results: valid
TF-IDF Validation
0.8697004608294931
No TF-IDF Validation
0.856752432155658
TF-IDF + handcrafted features - Validation
0.8689100102406554


Calculating BOWs: 100%|███████████████████| 5000/5000 [00:00<00:00, 6306.11it/s]


Training classifier with vocab size 2000 ...
BOW results: train
0.892561070244281
BOW results: valid
TF-IDF Validation
0.8767249103942651
No TF-IDF Validation
0.8541570660522273
TF-IDF + handcrafted features - Validation
0.870321300563236


In experiments here we modified the BOW to use TF-IDF that increased the balanced accuracy and added the same handcrafted features that worked best in the previous exercies. The results measured by balanced accuracy score as visible below. The handcrafted features did show a small improvement with smaller vocabulary, but when we increased its size the returns started to diminish. The vocabulary of 2000 words using TF-IDF and no handcrafted features achieved best balanced accuracy on validation set of 0.8767249103942651.

**Vocab 1000 words:**

TF-IDF Validation
0.855453149001536

No TF-IDF Validation
0.8479102662570404

TF-IDF + handcrafted features - Validation
0.8599174347158218

**Vocab 1500 words:**

TF-IDF Validation
0.8697004608294931

No TF-IDF Validation
0.856752432155658

TF-IDF + handcrafted features - Validation
0.8689100102406554

**Vocab 2000 words:**

TF-IDF Validation
0.8767249103942651

No TF-IDF Validation
0.8541570660522273

TF-IDF + handcrafted features - Validation
0.870321300563236


In [82]:
final_classifier_bow = LogisticRegression(solver='lbfgs', max_iter=1000)
vocab_final = create_vocab(dataset['train'], 1500)
token_to_id_final = create_token_to_id(vocab)
tfidf = TfidfTransformer()
train_bows = get_bows(dataset['train'], token_to_id)
train_tfidf = tfidf.fit_transform(train_bows)
final_classifier_bow.fit(train_tfidf, y_train)



Calculating BOWs: 100%|█████████████████| 20000/20000 [00:03<00:00, 6299.23it/s]


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

## 4. BERT Model

For the first part of this lab, we will be using a pre-trained BERT model from Huggingface, namely the [BERT Cased](https://huggingface.co/google-bert/bert-base-cased). You can read the original paper that introduced this model [here](https://aclanthology.org/N19-1423.pdf). This paper has been one of the most cited papers ever (currently having more than 100,000 citations).

We will specify the model name that can be found on the model's card on huggingface (revisit the first link). Make sure to check what other information Huggingface is offering (e.g. how to use the model, limitations, how to inference, etc.).

In [83]:
model_name = 'google-bert/bert-base-cased'

### 4.1 Tokenizer

The models on huggingface come with their own tokenizers. They are loaded separately from the models. We can use [AutoTokenizer](https://huggingface.co/docs/transformers/v4.40.2/en/model_doc/auto#transformers.AutoTokenizer)'s `from_pretrained()` method to load it.

Inspect the output: The loaded object is of `BertTokenizer` class. Check the documentation [here](https://huggingface.co/docs/transformers/en/model_doc/bert#transformers.BertTokenizer).

In [84]:
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
print(tokenizer)

BertTokenizer(name_or_path='google-bert/bert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


Next, let's see how we can use it to tokenize some text.

In [85]:
print(dataset['test'][0]['text'])
tokenized = tokenizer(dataset['test'][0]['text'], padding=True, return_tensors='pt')
print("---")
print(type(tokenized))
print("---")
print(tokenized)

Token indices sequence length is longer than the specified maximum sequence length for this model (927 > 512). Running this sequence through the model will result in indexing errors


I really like the movie ' s opening , when Col . Ted Masters realizes on his fighter radar that four enemy aircraft were approaching from about 10 o ' clock . The good news is that the movie does not mention at the very beginning that the colonel , along with a wingman fighter who was a lieutenant , was trying to do a " freedom of navigation " exercise along the eastern Meditteranean Sea , but went a little too past the restricted air space zone reserved for a rogue Middle eastern nation as they accidentially fly past it . I also like all of the intercutting on the colonel ' s fighter radar readouts and computer displays as the enemy aircraft aggressively picks the two American fighter pilots into an engagement for violating their airspace . That first dogfight immediately reminds me of the famous fighter pilot movie , " Top Gun . " From the waxing of the enemy bandits to the enemy aircraft ' s thirty - milimeter rounds that struck the colonel ' s jet engines and forcing the plane down

Examine the outputs: The tokenizer returned three things:
- `input_ids` - this is a PyTorch tensor ([https://pytorch.org/docs/stable/tensors.html](https://pytorch.org/docs/stable/tensors.html)) with the indices of our tokens. PyTorch tensors are similar to numpy arrays. They hold data in a multidimensional array or matrix. The difference is that PyTorch tensors can be placed and modified on the GPU which greatly improves the speed of execution.
- `token_type_ids` - this tensor holds the information about the index of the sentence. This has to do with the classification objective from the original paper, where two sentences were given and the model had to predict if they are connected. Because we only included a single sentence, we have only zeros here. We will not be concerned with it in this lab.
- `attention_mask` - holds the mask that the model will use to determine if the tokens in the `input_ids` are the real tokens or *padding*. Padding is a technique used to ensure that all input sequences have the same length. BERT (like many other NLP models) process data in batches and requires each sequence in a batch to have the same length, so sequences that are shorter than the maximum sequence length in the batch are padded with special tokens. In this case, because we only inputted a single sentence, the mask contains only ones. Later you will see examples where this is not the case.

Let's see how exactly the sentence was tokenized and how we can retrieve the original text. Notice that some words have been split into multiple tokens (remember when we discussed sub-word tokenization in class?). Also pay attention to the added special tokens, namely `CLS` and `SEP`:

The `[CLS]` token is a special classification token added at the beginning of every input sequence. It stands for "classification" (daah!) and is used by BERT to aggregate information from the entire sequence. The final hidden state corresponding to this token (after passing through the transformer layers) is used as the aggregate sequence representation for classification tasks. We will use this later in the lab!

The `[SEP]` token is used to separate different segments or sentences within the input sequence. It stands for "separator" (daaah again!).

In [86]:
print(tokenized['input_ids'].shape)
print("---")
print(tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0]))
print("---")
print(len(tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0])))
print("---")
print(tokenizer.decode(tokenized['input_ids'][0]))
print("---")
print(tokenizer.decode(tokenized['input_ids'][0], skip_special_tokens=True))

torch.Size([1, 927])
---
['[CLS]', 'I', 'really', 'like', 'the', 'movie', "'", 's', 'opening', ',', 'when', 'Col', '.', 'Ted', 'Masters', 'realizes', 'on', 'his', 'fighter', 'radar', 'that', 'four', 'enemy', 'aircraft', 'were', 'approaching', 'from', 'about', '10', 'o', "'", 'clock', '.', 'The', 'good', 'news', 'is', 'that', 'the', 'movie', 'does', 'not', 'mention', 'at', 'the', 'very', 'beginning', 'that', 'the', 'colonel', ',', 'along', 'with', 'a', 'wing', '##man', 'fighter', 'who', 'was', 'a', 'lieutenant', ',', 'was', 'trying', 'to', 'do', 'a', '"', 'freedom', 'of', 'navigation', '"', 'exercise', 'along', 'the', 'eastern', 'Me', '##dit', '##tera', '##nea', '##n', 'Sea', ',', 'but', 'went', 'a', 'little', 'too', 'past', 'the', 'restricted', 'air', 'space', 'zone', 'reserved', 'for', 'a', 'rogue', 'Middle', 'eastern', 'nation', 'as', 'they', 'accident', '##ially', 'fly', 'past', 'it', '.', 'I', 'also', 'like', 'all', 'of', 'the', 'inter', '##cut', '##ting', 'on', 'the', 'colonel', "

Tokenizer can process a list of sentences. This will create a batched output with tensor's first dimension corresponding to the batch size (the number of sentences we passed to the tokenizer). Examine the following cell and make sure it makes sense to you.

In [87]:
print(dataset['test'][0:3]['text'])
tokenized = tokenizer(dataset['test'][0:3]['text'], padding=True, return_tensors='pt')
print(tokenized)
print(tokenized['input_ids'].shape)
print(tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0]))
print(len(tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0])))
print(tokenizer.decode(tokenized['input_ids'][0]))
print(tokenizer.decode(tokenized['input_ids'][0], skip_special_tokens=True))

['I really like the movie \' s opening , when Col . Ted Masters realizes on his fighter radar that four enemy aircraft were approaching from about 10 o \' clock . The good news is that the movie does not mention at the very beginning that the colonel , along with a wingman fighter who was a lieutenant , was trying to do a " freedom of navigation " exercise along the eastern Meditteranean Sea , but went a little too past the restricted air space zone reserved for a rogue Middle eastern nation as they accidentially fly past it . I also like all of the intercutting on the colonel \' s fighter radar readouts and computer displays as the enemy aircraft aggressively picks the two American fighter pilots into an engagement for violating their airspace . That first dogfight immediately reminds me of the famous fighter pilot movie , " Top Gun . " From the waxing of the enemy bandits to the enemy aircraft \' s thirty - milimeter rounds that struck the colonel \' s jet engines and forcing the pla

<a name='e7'></a>
#### Exercise 7: Questions about the tokenizer

Answer the following questions:
- (1p) What is the size of the vocabulary?
- (2p) What are the special tokens apart from `[CLS]` and `[SEP]`? What are their functions?

In [88]:
print(tokenizer.vocab_size)
print(tokenizer.all_special_tokens)

28996
['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']


--- YOUR ANSWERS HERE

------------------------

For `google-bert/bert-base-cased`, the tokenizer vocabulary size is **28,996**.

Special tokens (besides `[CLS]` and `[SEP]`) are:

- **`[PAD]`**: used to pad shorter sequences so all inputs in a batch have the same length.
- **`[UNK]`**: used when a token is unknown / not in the tokenizer vocabulary.
- **`[MASK]`**: used in BERT’s masked-language-model objective; the model learns to predict masked tokens.

So in total, BERT’s main special tokens are `[CLS]`, `[SEP]`, `[PAD]`, `[UNK]`, and `[MASK]`.

------------------------


### 4.2 Loading the Model

In this section, we will load and examine the model. We will start with selecting the device we will place the model on. This will be a GPU (if one is available) or a CPU.

Google Colab offers free access to GPU, provided there is availability (also based on quotas which may vary based on your usage and the overall demand on Colab's resources). If you are working locally, then if you don't have a GPU, CPU will be selected. For the first parts of the assignment running on CPU might be okay but when we have to process the dataset a GPU will be necessary.

The following cell will select the device for us.

In [89]:
device = 'cpu'
if torch.cuda.is_available():
  device = 'cuda:0'
elif torch.backends.mps.is_available():
  device = torch.device("mps")
print(f'Device: {device}')

Device: mps


Now, let's load the model from huggingface and move it (slowly because it's heavy due to the large number of parameters) on the device from the previous cell (the methods `to()`).

In [90]:
model = transformers.AutoModel.from_pretrained(model_name)
print('loaded on device:', model.device)
model.to(device)
print('moved to device', model.device)
print(model)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded on device: cpu
moved to device mps:0
BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
           

When loading the model you might have seen the warning about some unexpected weights. This means that the model on huggingface has some additional weights that were downloaded, but our model does not use them. In essence, you can load the same weights (as linked by our `model_name`) to load to different but related models. In our case those would be `BertForMaskedLM` or `BertForNextSentencePrediction` instead of our `BertModel`, which is loaded automatically as the `AutoModel`. Below is a way to load the weights into a different model.

In [91]:
# transformers.BertForMaskedLM.from_pretrained(model_name)

Next, let's use BERT model for inference. We will tokenize the first sentence of our dataset and pass it to the model. We set `output_hidden_states` to `True` in order to have access to the hidden states of the model. Those represent the latent representations after embedding and transformer layers.

In [93]:
tokenized = tokenizer(
    dataset['test'][0]['text'],
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors='pt'
).to(device)
print(tokenized)
model_output = model(**tokenized, output_hidden_states=True)

{'input_ids': tensor([[  101,   146,  1541,  1176,  1103,  2523,   112,   188,  2280,   117,
          1165,  9518,   119,  6564,  6935, 10875,  1113,  1117,  6295,  7746,
          1115,  1300,  3437,  2163,  1127,  8320,  1121,  1164,  1275,   184,
           112,  4705,   119,  1109,  1363,  2371,  1110,  1115,  1103,  2523,
          1674,  1136,  4734,  1120,  1103,  1304,  2150,  1115,  1103,  9431,
           117,  1373,  1114,   170,  3092,  1399,  6295,  1150,  1108,   170,
          6403,   117,  1108,  1774,  1106,  1202,   170,   107,  4438,  1104,
         11167,   107,  6730,  1373,  1103,  2638,  2508, 17903, 17528, 25362,
          1179,  3017,   117,  1133,  1355,   170,  1376,  1315,  1763,  1103,
          7458,  1586,  2000,  4834,  9142,  1111,   170, 21010,  3089,  2638,
          3790,  1112,  1152,  4216, 21323,  4689,  1763,  1122,   119,   146,
          1145,  1176,  1155,  1104,  1103,  9455, 12734,  1916,  1113,  1103,
          9431,   112,   188,  6295,  

Examine the next cell and make sure everything makes sense to you. Consult the [documentation](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertModel.forward) in case of doubt.

In [94]:
print(list(model_output.keys()))
print()
print('pooler_output:')
print(type(model_output['pooler_output']))
print(model_output['pooler_output'].shape)
print()
print('hidden_states:')
print(type(model_output['hidden_states']))
print(len(model_output['hidden_states']))
print(type(model_output['hidden_states'][0]))
print(model_output['hidden_states'][0].shape)
print()
print('last_hidden_state:')
print(type(model_output['last_hidden_state']))
print(model_output['last_hidden_state'].shape)

['last_hidden_state', 'pooler_output', 'hidden_states']

pooler_output:
<class 'torch.Tensor'>
torch.Size([1, 768])

hidden_states:
<class 'tuple'>
13
<class 'torch.Tensor'>
torch.Size([1, 512, 768])

last_hidden_state:
<class 'torch.Tensor'>
torch.Size([1, 512, 768])


<a name='e8'></a>
#### Exercise 8: Questions about the Model

Examine the output of the previous cells. Answer the following questions:
- (1p) What is the number of transformer layers in this model?
- (1p) What is the dimension of the embeddings?
- (1p) What is the hidden size of the FFN in the transformer layer?
- (1p) What is the total number of parameters of the model (hint: check the `num_parameters()` method of the model)?
- (1p) How can you find the vocabulary size from the model?
- (1p) What is the length of the `hidden_states` in the output? Why?

In [95]:
print("num transformer layers:", len(model.encoder.layer))
print("embedding dim:", model.embeddings.word_embeddings.embedding_dim)
print("FFN hidden size:", model.encoder.layer[0].intermediate.dense.out_features)
print("total params:", model.num_parameters())
print("vocab size:", model.embeddings.word_embeddings.num_embeddings)


x = tokenizer(dataset['test'][0]['text'], return_tensors='pt', truncation=True).to(device)
y = model(**x, output_hidden_states=True)
print("len(hidden_states):", len(y.hidden_states))

num transformer layers: 12
embedding dim: 768
FFN hidden size: 3072
total params: 108310272
vocab size: 28996
len(hidden_states): 13


--- YOUR ANSWERS HERE

-----------------------

For `google-bert/bert-base-cased`:

- The model has **12** transformer layers.
- The embedding size is **768**.
- The FFN hidden size (intermediate size in each transformer block) is **3072**.
- Total number of parameters is **108,310,272** (about **108M**).
- You can find it from the model’s word-embedding layer: the number of embeddings (or rows in the embedding matrix) is the vocabulary size.
- The length of `hidden_states` is **13**.  
  Why: we get **1 output from embeddings + 12 outputs from the 12 transformer layers**.
-----------------------

## 5. BERT Sentence Embeddings

Having the model loaded and ready we can work on obtaining the sentence embeddings. During the last lab, you averaged the token embeddings. This time we will start with something else. Remember the CLS token? Its hidden representation is often used for classification as a representation of the whole sentence. We will do exactly that.

But first, we have to tokenize the dataset using BERT tokenizer.

<a name='e9'></a>
#### Exercise 9: BERT tokenizing examples

(5p) Fill in the following function to embed the examples (passed as a parameter) using the tokenizer (also a parameter). The function will tokenize a batch of examples, but the tokenizer can handle that, if you remember from the previous section.

In [96]:
def tokenize_text_bert(examples, tokenizer):
    """
    Tokenizes the `sentence` column from the batch of examples and returns the whole output of the tokenizer.
    Args:
        examples: a batch of examples
        tokenizer: the BERT tokenizer

    Returns: the tokenized `sentence` column (returns the whole output of the tokenizer)

    """
    ### YOUR CODE HERE
    tokenized_sentence = tokenizer(examples['text'], truncation=True)


    ### YOUR CODE ENDS HERE
    return tokenized_sentence


In [97]:
dataset_tokenized_bert = dataset.map(tokenize_text_bert,
                                     fn_kwargs={'tokenizer': tokenizer},
                                     batched=True,
                                     remove_columns=dataset['train'].column_names,)
print(dataset_tokenized_bert)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
})


<a name='e10'></a>
#### Exercise 10: BERT sentence embeddings by the CLS token

(5p) Implement the following function which calculates the sentence embeddings based on the model output (passed to the function as a parameter). It should take the embedding of the CLS token of last layer.

In [98]:
def calculate_cls_embeddings(input_batch, model_output):
    """
    Calculates the sentence embeddings of a batch of sentences as the last-layer representation of the CLS token.
    Args:
        input_batch: tokenized batch of sentences (as returned by the tokenizer), contains `input_ids`, `token_type_ids`, and `attention_mask` tensors
        model_output: the output of the model given the `input_batch`, contains `last_hidden_state`, `pooler_output`, `hidden_states` tensors

    Returns: tensor of the hidden states of the CLS token (from the last layer) for each example in the batch

    """

    ### YOUR CODE HERE
    sentence_embeddings = model_output['last_hidden_state'][:, 0, :]


    ### YOUR CODE ENDS HERE

    return sentence_embeddings

In [99]:
text = "The weather is nice today."
tokenized = tokenizer(text, padding=True, return_tensors='pt').to(device)
print(tokenized)
model_output = model(**tokenized, output_hidden_states=True)
print(model_output['last_hidden_state'].shape)
sentence_embedding = calculate_cls_embeddings(tokenized, model_output)
print(sentence_embedding.shape)

{'input_ids': tensor([[ 101, 1109, 4250, 1110, 3505, 2052,  119,  102]], device='mps:0'), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]], device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}
torch.Size([1, 8, 768])
torch.Size([1, 768])


In [100]:
def embed_dataset(dataset, model, sentence_embedding_fn, batch_size=8):
    data_collator = transformers.DataCollatorWithPadding(tokenizer)
    data_loader = DataLoader(dataset, batch_size=batch_size, collate_fn=data_collator)
    sentence_embeddings = []
    with torch.no_grad():
        for batch in tqdm.tqdm(data_loader):
            batch.to(device)
            model_output = model(**batch, output_hidden_states=True)
            batch_sentence_embeddings = sentence_embedding_fn(batch, model_output)
            sentence_embeddings.append(batch_sentence_embeddings.detach().cpu())

    sentence_embeddings = torch.concat(sentence_embeddings, dim=0)
    return sentence_embeddings

In [101]:
bert_cls_train = embed_dataset(dataset_tokenized_bert['train'], model, calculate_cls_embeddings)
print(bert_cls_train.shape)

bert_cls_valid = embed_dataset(dataset_tokenized_bert['validation'], model, calculate_cls_embeddings)
print(bert_cls_valid.shape)

100%|███████████████████████████████████████| 2500/2500 [33:26<00:00,  1.25it/s]


torch.Size([20000, 768])


100%|█████████████████████████████████████████| 625/625 [09:50<00:00,  1.06it/s]

torch.Size([5000, 768])


In [102]:
classifier = LogisticRegression(solver='lbfgs', max_iter=1000)
print('Training classifier...')
classifier.fit(bert_cls_train, y_train)

Training classifier...


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [103]:
print('BERT train')
pred_train = classifier.predict(bert_cls_train)
print(accuracy_score(y_train, pred_train))

print('BERT valid')
pred_valid = classifier.predict(bert_cls_valid)
print(accuracy_score(y_valid, pred_valid))

BERT train
0.86425
BERT valid
0.8456




```
# This is formatted as code
```

You can test the model on the evaluation metric of your choice:

In [104]:
#### YOUR CODE HERE

from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, balanced_accuracy_score, roc_auc_score

valid_proba = classifier.predict_proba(bert_cls_valid)[:, 1]

# validation metrics
print("Validation precision:", precision_score(y_valid, pred_valid))
print("Validation recall:", recall_score(y_valid, pred_valid))
print("Validation F1:", f1_score(y_valid, pred_valid))
print("Validation balanced accuracy:", balanced_accuracy_score(y_valid, pred_valid))
print("Validation ROC-AUC:", roc_auc_score(y_valid, valid_proba))

print("\nConfusion matrix (validation):")
print(confusion_matrix(y_valid, pred_valid))

#### YOUR CODE ENDS HERE

Validation precision: 0.8552845528455284
Validation recall: 0.834920634920635
Validation F1: 0.8449799196787149
Validation balanced accuracy: 0.8456861239119304
Validation ROC-AUC: 0.9220006080389144

Confusion matrix (validation):
[[2124  356]
 [ 416 2104]]


<a name='e11'></a>
#### Exercise 11: BERT Sentence embeddings by averaging tokens

(5p) Implement embedding sentences by averaging the hidden representations of tokens. Make sure to ignore the special and padding tokens. The padding tokens are indicated by the attention mask. You can find the other special tokens using the tokenizer's attributes such as `tokenizer.sep_token_id`. The function accepts the `layer` parameter. Typically, you would use the hidden representations of the last layer, it might be beneficial for some tasks to use previous layers or an averaged representations of multiple layers.

In [105]:
### YOUR CODE HERE

def calculate_sentence_embeddings(input_batch, model_output, layer=-1):
    """
    Calculates the sentence embeddings of a batch of sentences as a mean of token representations.
    The representations are taken from the layer of the index provided as a `layer` parameter.
    Args:
        input_batch: tokenized batch of sentences (as returned by the tokenizer), contains `input_ids`, `token_type_ids`, and `attention_mask` tensors
        model_output: the output of the model given the `input_batch`, contains `last_hidden_state`, `pooler_output`, `hidden_states` tensors
        layer: specifies the layer of the hidden states that are used to calculate sentence embedding

    Returns: tensor of the averaged hidden states (from the specified layer) for each example in the batch

    """
    attention_mask = input_batch['attention_mask']
    hidden_states = model_output['hidden_states'][layer]

    ### YOUR CODE HERE
    input_ids = input_batch['input_ids']
    special_ids = {
      tokenizer.cls_token_id,
      tokenizer.sep_token_id,
      tokenizer.pad_token_id,
    }

    valid_mask = attention_mask.bool()
    for token_id in special_ids:
        if token_id is not None:
            valid_mask = valid_mask & (input_ids != token_id)

    valid_mask = valid_mask.unsqueeze(-1)
    token_sum = (hidden_states * valid_mask).sum(dim=1)
    token_count = valid_mask.sum(dim=1).clamp(min=1)
    sentence_embeddings = token_sum / token_count


    ### YOUR CODE ENDS HERE

    return sentence_embeddings

We can test it here:


In [106]:
text = "The weather is nice today."
tokenized = tokenizer(text, padding=True, return_tensors='pt').to(device)
print(tokenized)
model_output = model(**tokenized, output_hidden_states=True)
print(model_output['last_hidden_state'].shape)
sentence_embedding = calculate_sentence_embeddings(tokenized, model_output)
print(sentence_embedding.shape)

{'input_ids': tensor([[ 101, 1109, 4250, 1110, 3505, 2052,  119,  102]], device='mps:0'), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]], device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}
torch.Size([1, 8, 768])
torch.Size([1, 768])


We will embed the sentences and evaluate the model on the `validation` subset.


In [107]:
bert_sentence_train = embed_dataset(dataset_tokenized_bert['train'], model, calculate_sentence_embeddings)
print(bert_cls_train.shape)

bert_sentence_valid = embed_dataset(dataset_tokenized_bert['validation'], model, calculate_sentence_embeddings)
print(bert_cls_valid.shape)

100%|███████████████████████████████████████| 2500/2500 [37:32<00:00,  1.11it/s]


torch.Size([20000, 768])


100%|█████████████████████████████████████████| 625/625 [09:21<00:00,  1.11it/s]

torch.Size([5000, 768])


In [108]:
classifier = LogisticRegression(solver='lbfgs', max_iter=1000)
print('Training classifier...')
classifier.fit(bert_sentence_train, y_train)

Training classifier...


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [109]:
print('BERT train')
pred_train = classifier.predict(bert_sentence_train)
print(accuracy_score(y_train, pred_train))

print('BERT valid')
pred_valid = classifier.predict(bert_sentence_valid)
print(accuracy_score(y_valid, pred_valid))

BERT train
0.89715
BERT valid
0.8836


Test the model on the evaluation metric of your choice:


In [110]:
#### YOUR CODE HERE
# from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, balanced_accuracy_score, roc_auc_score

# valid_proba = classifier.predict_proba(bert_sentence_valid)[:, 1]

# print("Validation precision:", precision_score(y_valid, pred_valid))
# print("Validation recall:", recall_score(y_valid, pred_valid))
# print("Validation F1:", f1_score(y_valid, pred_valid))
# print("Validation balanced accuracy:", balanced_accuracy_score(y_valid, pred_valid))
# print("Validation ROC-AUC:", roc_auc_score(y_valid, valid_proba))
# print("\nConfusion matrix (validation):")
# print(confusion_matrix(y_valid, pred_valid))

# Reusing the code for prediction evaluation we have written earlier.
evaluate(y_valid, pred_valid, "Validation")
valid_proba = classifier.predict_proba(bert_sentence_valid)[:, 1]
print("Validation ROC-AUC:", roc_auc_score(y_valid, valid_proba))

#### YOUR CODE ENDS HERE

Validation
Accuracy: 0.8836
F1: 0.8835990874168453
Balanced Accuracy: 0.8836341525857655
Confusion Matrix:
 [[2202  278]
 [ 304 2216]]

Per-class report:
               precision    recall  f1-score   support

           0     0.8787    0.8879    0.8833      2480
           1     0.8885    0.8794    0.8839      2520

    accuracy                         0.8836      5000
   macro avg     0.8836    0.8836    0.8836      5000
weighted avg     0.8837    0.8836    0.8836      5000


Validation ROC-AUC: 0.9511355926779315


In [111]:
final_classifier_bert = classifier

## 6. Testing all methods

In this last section, you will bering together all of what you have done so far in this lab. First, you will find the best classifier. Next, you will evaluate all the models you created so far.

<a name='e12'></a>
#### Exercise 12: Find the best classifier for the models

(5p) Basically, do what the title of the exercise says. Evaluate on the `validation` subset. Try at least two other classifiers (apart from the logistic regression). Comment on the results.

In [112]:
#### YOUR CODE HERE
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier

def compare_models(Xtr, ytr, Xva, yva):
    models = {
        "LogReg": LogisticRegression(max_iter=2000),
        "LinearSVC": LinearSVC(),
        "SGD-hinge": SGDClassifier(loss="hinge", max_iter=2000, random_state=0),
    }
    for name, clf in models.items():
        clf.fit(Xtr, ytr)
        pred = clf.predict(Xva)
        print(
            f"{name:10s}  "
            f"accuracy={accuracy_score(yva, pred):.4f}  "
            f"F1={f1_score(yva, pred, average='macro'):.4f}  "
            f"balanced accuracy={balanced_accuracy_score(yva, pred):.4f}"
        )

compare_models(bert_sentence_train, y_train, bert_sentence_valid, y_valid)
#asked AI what classifiers are the standard for text classification
#### YOUR CODE ENDS HERE

LogReg      accuracy=0.8836  F1=0.8836  balanced accuracy=0.8836
LinearSVC   accuracy=0.8786  F1=0.8786  balanced accuracy=0.8787
SGD-hinge   accuracy=0.8794  F1=0.8792  balanced accuracy=0.8792




```
# This is formatted as code
```

--- YOUR ANSWERS HERE

Logistic regression achieved the highest validation performance (on all metrics f1/accuracy/balanced accuracy). LinearSVC performed slightly worse, while SGDClassifier with hinge loss performed worse than the two. Since BERT embeddings are already structured, logistic regression effectively separates the classes.

<a name='e13'></a>
#### Exercise 13: Evaluating methods on the test set

(10p) Test the models you implemented on the test subset:
- Hand-crafted features,
- BOW,
- BERT model based on the CLS token.

You have the models trained already, so only do evaluation.

Evaluate the performance using the metric(s) of your choice. Make sure to discuss the results. Which model performed best? Is this what you expected?

In [113]:
#### YOUR CODE HERE
X_test_custom = calculate_features_dataset(dataset['test'], calculate_features_enhanced_v2)

X_test_bow = get_bows(dataset['test'], token_to_id_final)
X_test_bow = tfidf.transform(X_test_bow)


Calculating BOWs: 100%|█████████████████| 25000/25000 [00:05<00:00, 4829.09it/s]


In [114]:
X_test_bert = embed_dataset(dataset_tokenized_bert['test'], model, calculate_sentence_embeddings)

100%|███████████████████████████████████████| 3125/3125 [42:07<00:00,  1.24it/s]


In [115]:
y_test = dataset['test']['label']

In [116]:
y_pred_custom = final_classifier_handcrafted.predict(X_test_custom)

y_pred_bow = final_classifier_bow.predict(X_test_bow)
y_pred_bert = final_classifier_bert.predict(X_test_bert)

evaluate(y_test, y_pred_custom, "Custom features")
evaluate(y_test, y_pred_bow, "BOW")
evaluate(y_test, y_pred_bert, "BERT")

Custom features
Accuracy: 0.74076
F1: 0.7407599896303996
Balanced Accuracy: 0.74076
Confusion Matrix:
 [[9262 3238]
 [3243 9257]]

Per-class report:
               precision    recall  f1-score   support

           0     0.7407    0.7410    0.7408     12500
           1     0.7409    0.7406    0.7407     12500

    accuracy                         0.7408     25000
   macro avg     0.7408    0.7408    0.7408     25000
weighted avg     0.7408    0.7408    0.7408     25000


BOW
Accuracy: 0.8734
F1: 0.8733939372914388
Balanced Accuracy: 0.8734
Confusion Matrix:
 [[10831  1669]
 [ 1496 11004]]

Per-class report:
               precision    recall  f1-score   support

           0     0.8786    0.8665    0.8725     12500
           1     0.8683    0.8803    0.8743     12500

    accuracy                         0.8734     25000
   macro avg     0.8735    0.8734    0.8734     25000
weighted avg     0.8735    0.8734    0.8734     25000


BERT
Accuracy: 0.8832
F1: 0.8831954519112284
Balanced 

In [117]:
### YOUR CODE ENDS HERE

--- YOUR ANSWERS HERE

Our handcrafted features vector performed the worst as expected.
BOW representation and BERT performed much closer together, altough our best BERT model outperformed the BOW achieving around 88% balanced accuracy vs around 87% in case of BOW implementation.

BERT ranked best as we expected because it has a deep understanding of context and the model was pre-trained on massive data. Because we did not fine-tune it for this task precisely enough and due to limited computational resources we did not achieve higher balanced accuracy and other metrics.

BOW performed decently as well because it catches basic word patterns but misses the "flow", order of sentences and is limited to only our trainig data from the IMDB reviews. We extracted as much as we could from it.

Hand crafted vector was the worst out of all. It is too basic. It correctly identifies positive & negative words from our lists but can't handle nuances of human language. It acheived respectable accuracy though of around 75%.

BERT takes way more computing power and time to run while BOW is very fast with a good enought results so it might be a good choice for some tasks.
